In [32]:
import os
import subprocess
import sys
from pathlib import Path

# Calculamos la ruta
SRC = Path.cwd().parent / "src"

def run(script, *args):
    """Runs a script from src/, showing its output live.
    Interrupting the cell (■ button / Interrupt) kills the process and all its children."""
    cmd = [sys.executable, "-u", script, *args]   # -u: unbuffered output
    flags = subprocess.CREATE_NEW_PROCESS_GROUP if os.name == "nt" else 0
    proc = subprocess.Popen(
        cmd,
        cwd=str(SRC),                             # SOLUCIÓN PARA WINDOWS: Forzar conversión a string
        stdin=subprocess.PIPE,                    # minimal.py expects input(); keep stdin open
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,                 # logging goes to stderr: merge it with stdout
        text=True,
        bufsize=1,
        creationflags=flags,
        start_new_session=(os.name != "nt"),
    )
    try:
        for line in proc.stdout:
            print(line, end="", flush=True)
        proc.wait()
    except KeyboardInterrupt:
        print("\n[Interrupted: stopping the process...]")
    finally:
        if proc.poll() is None:
            if os.name == "nt":
                subprocess.run(["taskkill", "/F", "/T", "/PID", str(proc.pid)],
                               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            else:
                import signal
                os.killpg(proc.pid, signal.SIGKILL)
            proc.wait()
        print(f"[Process finished, exit code {proc.returncode}]")

# Práctica: Supresión de Eco Acústico
## Miembros del grupo:
Javier Rodriguez Gallegos

Marcos Gallego Pérez

Rocío Lu Gonzalez Pérez

Andrés Xandri López

## 1. Implementación del Módulo (`feedback_supression.py`)
La siguiente celda genera el módulo en la carpeta `src/` implementando el algoritmo adaptativo NLMS y exponiendo los parámetros por línea de comandos solicitados.

```bash
%%writefile src/feedback_supression.py
import argparse
import numpy as np
import threading
import sys
import buffer
import minimal

class NLMS:
    """Implementación matemática del algoritmo NLMS (Sección 7.3)"""
    def __init__(self, L=1024, mu=0.5, eps=1e-6):
        self.L = L
        self.mu = mu
        self.eps = eps
        self.w = np.zeros(L)
        self.hist = np.zeros(L - 1)

    def process(self, x, d):
        # Aseguramos que el tamaño de las señales se adapte dinámicamente a L
        if len(x) != self.L:
            self.L = len(x)
            self.w = np.zeros(self.L)
            self.hist = np.zeros(self.L - 1)
            
        x_float = x.astype(np.float32)
        d_float = d.astype(np.float32)
        
        buf = np.concatenate((self.hist, x_float))
        e = np.empty_like(d_float)
        
        for n in range(len(d_float)):
            u = buf[n:n + self.L][::-1]
            e[n] = d_float[n] - self.w @ u
            self.w += (self.mu * e[n] / (self.eps + u @ u)) * u
            
        self.hist = buf[-(self.L - 1):]
        return e.astype(d.dtype)

# Reutilizamos el parser original de la jerarquía de intercomunicación
parser = minimal.parser
parser.add_argument('--echo-suppression', action='store_true', default=False)
parser.add_argument('--nlms-mu', type=float, default=0.5)

args = parser.parse_args()
minimal.args = args
buffer.args = args

BaseClass = getattr(buffer, 'Buffering', getattr(buffer, 'Buffer', None))

class Feedback_supression(BaseClass):
    """Subclase corregida para inyectar bloques de audio reales al filtro"""
    def __init__(self):
        super().__init__()
        
        # Inicializamos un bloque vacío con ceros para que record() tenga señal 
        # de referencia desde el segundo cero y el filtro empiece a entrenar de inmediato.
        # Intentamos obtener frames_per_chunk del framework, si no usamos 1024 por defecto.
        chunk_cells = getattr(args, 'frames_per_chunk', 1024)
        self.last_played_chunk = np.zeros(chunk_cells, dtype=np.int16)
        
        if args.echo_suppression:
            # Inicializamos el filtro adaptativo adaptado al framework
            self.nlms = NLMS(L=chunk_cells, mu=args.nlms_mu)
            self.interactive = threading.Thread(target=self.interactive_loop, daemon=True)
            self.interactive.start()
        else:
            self.nlms = None

    def interactive_loop(self):
        print("\n[Control Real-Time] Introduce '+' para subir mu, '-' para bajar mu.", flush=True)
        while True:
            char = sys.stdin.read(1)
            if char == '+':
                self.nlms.mu = min(2.0, self.nlms.mu + 0.05)
                print(f"\n[Real-Time] mu aumentado a: {self.nlms.mu:.2f}", flush=True)
            elif char == '-':
                self.nlms.mu = max(0.01, self.nlms.mu - 0.05)
                print(f"\n[Real-Time] mu disminuido a: {self.nlms.mu:.2f}", flush=True)

    def record(self):
        """Captura el bloque del micrófono y le resta el eco del altavoz"""
        recorded_chunk = super().record()
        
        # El filtro ahora siempre se ejecutará porque last_played_chunk nunca es None
        if self.nlms is not None:
            # Si el bloque del micrófono viene en estéreo o plano, lo aplanamos para el filtro lineal
            d = recorded_chunk.flatten()
            x = self.last_played_chunk.flatten()
            
            clean_flat = self.nlms.process(x, d)
            # Devolvemos el audio limpio reconstruyendo la forma original del bloque
            return clean_flat.reshape(recorded_chunk.shape)
            
        return recorded_chunk

    def play(self, chunk):
        """Almacena inmediatamente el bloque que se va a reproducir"""
        # Guardamos una copia del bloque antes de enviarlo al hardware del altavoz
        self.last_played_chunk = chunk.copy()
        super().play(chunk)

if __name__ == "__main__":
    intercom = Feedback_supression()
    intercom.run()

```

In [33]:
%%writefile src/feedback_supression.py
import argparse
import numpy as np
import threading
import sys
import buffer
import minimal

class NLMS:
    """Implementación matemática del algoritmo NLMS (Sección 7.3)"""
    def __init__(self, L=1024, mu=0.5, eps=1e-6):
        self.L = L
        self.mu = mu
        self.eps = eps
        self.w = np.zeros(L)
        self.hist = np.zeros(L - 1)

    def process(self, x, d):
        # Aseguramos que el tamaño de las señales se adapte dinámicamente a L
        if len(x) != self.L:
            self.L = len(x)
            self.w = np.zeros(self.L)
            self.hist = np.zeros(self.L - 1)
            
        x_float = x.astype(np.float32)
        d_float = d.astype(np.float32)
        
        buf = np.concatenate((self.hist, x_float))
        e = np.empty_like(d_float)
        
        for n in range(len(d_float)):
            u = buf[n:n + self.L][::-1]
            e[n] = d_float[n] - self.w @ u
            self.w += (self.mu * e[n] / (self.eps + u @ u)) * u
            
        self.hist = buf[-(self.L - 1):]
        return e.astype(d.dtype)

# Reutilizamos el parser original de la jerarquía de intercomunicación
parser = minimal.parser
parser.add_argument('--echo-suppression', action='store_true', default=False)
parser.add_argument('--nlms-mu', type=float, default=0.5)

args = parser.parse_args()
minimal.args = args
buffer.args = args

BaseClass = getattr(buffer, 'Buffering', getattr(buffer, 'Buffer', None))

class Feedback_supression(BaseClass):
    """Subclase corregida para inyectar bloques de audio reales al filtro"""
    def __init__(self):
        super().__init__()
        
        # Inicializamos un bloque vacío con ceros para que record() tenga señal 
        # de referencia desde el segundo cero y el filtro empiece a entrenar de inmediato.
        # Intentamos obtener frames_per_chunk del framework, si no usamos 1024 por defecto.
        chunk_cells = getattr(args, 'frames_per_chunk', 1024)
        self.last_played_chunk = np.zeros(chunk_cells, dtype=np.int16)
        
        if args.echo_suppression:
            # Inicializamos el filtro adaptativo adaptado al framework
            self.nlms = NLMS(L=chunk_cells, mu=args.nlms_mu)
            self.interactive = threading.Thread(target=self.interactive_loop, daemon=True)
            self.interactive.start()
        else:
            self.nlms = None

    def interactive_loop(self):
        print("\n[Control Real-Time] Introduce '+' para subir mu, '-' para bajar mu.", flush=True)
        while True:
            char = sys.stdin.read(1)
            if char == '+':
                self.nlms.mu = min(2.0, self.nlms.mu + 0.05)
                print(f"\n[Real-Time] mu aumentado a: {self.nlms.mu:.2f}", flush=True)
            elif char == '-':
                self.nlms.mu = max(0.01, self.nlms.mu - 0.05)
                print(f"\n[Real-Time] mu disminuido a: {self.nlms.mu:.2f}", flush=True)

    def record(self):
        """Captura el bloque del micrófono y le resta el eco del altavoz"""
        recorded_chunk = super().record()
        
        # El filtro ahora siempre se ejecutará porque last_played_chunk nunca es None
        if self.nlms is not None:
            # Si el bloque del micrófono viene en estéreo o plano, lo aplanamos para el filtro lineal
            d = recorded_chunk.flatten()
            x = self.last_played_chunk.flatten()
            
            clean_flat = self.nlms.process(x, d)
            # Devolvemos el audio limpio reconstruyendo la forma original del bloque
            return clean_flat.reshape(recorded_chunk.shape)
            
        return recorded_chunk

    def play(self, chunk):
        """Almacena inmediatamente el bloque que se va a reproducir"""
        # Guardamos una copia del bloque antes de enviarlo al hardware del altavoz
        self.last_played_chunk = chunk.copy()
        super().play(chunk)

if __name__ == "__main__":
    intercom = Feedback_supression()
    intercom.run()

Writing src/feedback_supression.py


### Evaluación experimental y limitaciones técnicas del sistema

En las pruebas realizadas en tiempo real con **InterCom**, el algoritmo **NLMS** (*Normalized Least Mean Squares*) logra converger en pocos segundos y suprimir eficazmente el eco acústico cuando el volumen de los altavoces se mantiene en un nivel bajo o moderado. Sin embargo, al superar cierto umbral de volumen, la cancelación pierde efectividad y reaparece el acoplamiento por tres limitaciones físicas y algorítmicas:

* **No linealidades del hardware (saturación):** El filtro adaptativo NLMS modela el entorno acústico como un sistema lineal mediante un filtro FIR ($\hat{\mathbf{s}} = \mathbf{h} * \mathbf{s}$). A volúmenes altos, los altavoces integrados vibran de forma no lineal generando distorsión armónica y el micrófono puede sufrir recorte (*clipping* al saturar el rango de `int16`), comportamientos que ningún filtro lineal puede predecir ni restar.
* **Cola de reverberación frente a la longitud del filtro ($L$):** Como se indica en la Sección 7.2 del guion, el número de coeficientes debe cubrir el retardo acústico y la reverberación ($L \approx f_s T$). Al subir mucho el volumen, los rebotes tardíos en las paredes de la habitación adquieren suficiente energía para alcanzar el micrófono fuera de la ventana temporal que cubre $L$, quedando ese eco residual sin cancelar.
* **Latencia del sistema y *Double-Talk*:** El desfase entre los hilos de reproducción (`play`) y captura (`record`) introduce ceros iniciales en la respuesta al impulso $\mathbf{h}$ que consumen parte de los coeficientes útiles del filtro. Además, al no contar con un detector de doble habla (*Double-Talk Detector*, Sección 7.4), cualquier pico de ruido o voz local a alto volumen perturba temporalmente la actualización de los pesos $\mathbf{w}$.

### Ejemplo de ejecución

Para comprobar el funcionamiento en local (*loopback*, donde el audio capturado por el micrófono se reproduce directamente por nuestros propios altavoces), se ejecuta el módulo activando la supresión de eco:

```bash
python .\feedback_supression.py --echo-suppression
```

Durante la ejecución en tiempo real, es posible ajustar dinámicamente el paso de aprendizaje ($\mu$) desde la terminal pulsando las teclas `+` o `-` para encontrar el equilibrio óptimo entre velocidad de convergencia y estabilidad del filtro.